<a href="https://colab.research.google.com/github/kawastony/Quadratic-Mechanism-Lens/blob/main/TIFA_Paper_B_Calc_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# TIFA / cosine-quintessence + DESI DR1 BAO likelihood
# One self-contained script
#
# Conventions:
#   Mpl = 1
#   H0 = 1 today
#   rho_c0 = 3
#   rho_m(a) = 3 Omega_m a^-3
#   rho_r(a) = 3 Omega_r a^-4
#
# Friedmann:
#   H^2 = (rho_m + rho_r + V) / (3 - 0.5 phip^2)
#
# Scalar:
#   V(phi) = L4 * [1 - cos(phi/fE)]
#
# This script:
#   1) Solves the KG+Friedmann background
#   2) Shoots L4 so H(a=1)=1
#   3) Computes H(z), q(z), w_phi(z)
#   4) Builds DESI BAO observables
#   5) Evaluates chi^2 using the 12-point covariance
#   6) Scans phi_ini if desired
# ============================================================

import numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq

# ============================================================
# SECTION A — COSMOLOGICAL PARAMETERS
# ============================================================

Omega_m = 0.315
Omega_r = 9.0e-5
Omega_phi0_target = 1.0 - Omega_m - Omega_r
Omega_L = 1.0 - Omega_m - Omega_r

fE = 2.0           # decay constant in reduced Planck units
z_ini_default = 1000.0

# Sound horizon:
# IMPORTANT:
# DESI BAO observables are D/rd or H*rd in physical analyses.
# In your previous setup the data vector is already in D/rd and DH/rd,
# so here we compute dimensionless D_M/rd etc. by introducing rd_dimless.
#
# Since H0=1 units are abstract, rd_dimless acts as the calibration parameter.
# If you want Planck-anchored rd, set this accordingly.
# If you want BAO-only with free H0*rd, this becomes a nuisance parameter.
#
# For now we keep rd_dimless explicit.
rd_dimless_default = 0.0333   # placeholder dimensionless rd * H0/c-type scale

c_light = 1.0  # in these dimensionless units we absorb c consistently into rd choice


# ============================================================
# SECTION B — SCALAR POTENTIAL
# ============================================================

def V(phi, L4):
    return L4 * (1.0 - np.cos(phi / fE))

def dV_dphi(phi, L4):
    return (L4 / fE) * np.sin(phi / fE)

def d2V_dphi2(phi, L4):
    return (L4 / fE**2) * np.cos(phi / fE)


# ============================================================
# SECTION C — BACKGROUND DENSITIES
# ============================================================

def rho_m_of_a(a):
    return 3.0 * Omega_m / a**3

def rho_r_of_a(a):
    return 3.0 * Omega_r / a**4


# ============================================================
# SECTION D — EXACT FRIEDMANN / FLUID QUANTITIES
# ============================================================

def H_sq(a, phi, phip, L4):
    denom = 3.0 - 0.5 * phip**2
    if np.any(denom <= 0):
        return np.nan
    return (rho_m_of_a(a) + rho_r_of_a(a) + V(phi, L4)) / denom

def scalar_energy_pressure(a, phi, phip, L4):
    H2 = H_sq(a, phi, phip, L4)
    K = 0.5 * H2 * phip**2
    VV = V(phi, L4)
    rho_phi = K + VV
    p_phi = K - VV
    return rho_phi, p_phi, K, VV, H2

def w_phi(a, phi, phip, L4):
    rho_phi, p_phi, _, _, _ = scalar_energy_pressure(a, phi, phip, L4)
    return p_phi / rho_phi

def w_total(a, phi, phip, L4):
    rho_phi, p_phi, _, _, _ = scalar_energy_pressure(a, phi, phip, L4)
    rho_tot = rho_m_of_a(a) + rho_r_of_a(a) + rho_phi
    p_tot = rho_r_of_a(a) / 3.0 + p_phi
    return p_tot / rho_tot

def q_of_state(a, phi, phip, L4):
    return 0.5 * (1.0 + 3.0 * w_total(a, phi, phip, L4))

def Hprime_over_H(a, phi, phip, L4):
    wt = w_total(a, phi, phip, L4)
    return -1.5 * (1.0 + wt)


# ============================================================
# SECTION E — KG EQUATION IN N = ln a
# ============================================================

def rhs_background(N, y, L4):
    phi, phip = y
    a = np.exp(N)

    H2 = H_sq(a, phi, phip, L4)
    if not np.isfinite(H2) or H2 <= 0:
        return [np.nan, np.nan]

    HpH = Hprime_over_H(a, phi, phip, L4)
    phipp = -(3.0 + HpH) * phip - dV_dphi(phi, L4) / H2
    return [phip, phipp]


# ============================================================
# SECTION F — BACKGROUND SOLVER
# ============================================================

def integrate_background(phi_ini,
                         L4,
                         z_ini=z_ini_default,
                         phip_ini=0.0,
                         rtol=1e-9,
                         atol=1e-11,
                         dense_output=True,
                         method='RK45'):
    N_ini = np.log(1.0 / (1.0 + z_ini))
    N_end = 0.0

    sol = solve_ivp(
        fun=lambda N, y: rhs_background(N, y, L4),
        t_span=(N_ini, N_end),
        y0=[phi_ini, phip_ini],
        dense_output=dense_output,
        rtol=rtol,
        atol=atol,
        method=method
    )
    return sol

def today_diagnostics(sol, L4):
    if (not sol.success) or (sol.sol is None):
        return None

    phi0, phip0 = sol.sol(0.0)
    a0 = 1.0

    rho_phi, p_phi, K0, V0, H20 = scalar_energy_pressure(a0, phi0, phip0, L4)
    rho_m0 = rho_m_of_a(a0)
    rho_r0 = rho_r_of_a(a0)
    rho_tot0 = rho_m0 + rho_r0 + rho_phi
    p_tot0 = rho_r0 / 3.0 + p_phi

    return {
        "phi0": float(phi0),
        "phip0": float(phip0),
        "H0": float(np.sqrt(H20)),
        "H0_sq": float(H20),
        "rho_m0": float(rho_m0),
        "rho_r0": float(rho_r0),
        "rho_phi0": float(rho_phi),
        "p_phi0": float(p_phi),
        "rho_tot0": float(rho_tot0),
        "K0": float(K0),
        "V0": float(V0),
        "wphi0": float(p_phi / rho_phi),
        "wtot0": float(p_tot0 / rho_tot0),
        "q0": float(0.5 * (1.0 + 3.0 * p_tot0 / rho_tot0)),
        "Omega_m0_eff": float(rho_m0 / rho_tot0),
        "Omega_r0_eff": float(rho_r0 / rho_tot0),
        "Omega_phi0_eff": float(rho_phi / rho_tot0),
        "closure_error": float(rho_tot0 - 3.0 * H20)
    }

def H0_minus_1_for_root(L4, phi_ini, z_ini=z_ini_default, phip_ini=0.0):
    sol = integrate_background(phi_ini, L4, z_ini=z_ini, phip_ini=phip_ini)
    if (not sol.success) or (sol.sol is None):
        return np.nan
    diag = today_diagnostics(sol, L4)
    if diag is None or not np.isfinite(diag["H0"]):
        return np.nan
    return diag["H0"] - 1.0

def shoot_L4_for_H0_eq_1(phi_ini,
                         L4_min=1e-8,
                         L4_max=1e3,
                         z_ini=z_ini_default,
                         phip_ini=0.0):
    def f(L4):
        return H0_minus_1_for_root(L4, phi_ini, z_ini=z_ini, phip_ini=phip_ini)

    a = L4_min
    b = L4_max
    fa = f(a)
    fb = f(b)

    for _ in range(40):
        if np.isfinite(fa) and np.isfinite(fb) and fa * fb < 0:
            break
        a /= 10.0
        b *= 10.0
        fa = f(a)
        fb = f(b)
    else:
        raise RuntimeError("Could not bracket root for L4.")

    L4_star = brentq(f, a, b, xtol=1e-12, rtol=1e-10, maxiter=200)
    sol_star = integrate_background(phi_ini, L4_star, z_ini=z_ini, phip_ini=phip_ini)

    if not sol_star.success:
        raise RuntimeError("Integration failed after L4 shooting.")

    return L4_star, sol_star, today_diagnostics(sol_star, L4_star)


# ============================================================
# SECTION G — REDSHIFT-SPACE DIAGNOSTICS
# ============================================================

def state_at_z(z, sol, L4):
    a = 1.0 / (1.0 + z)
    N = np.log(a)
    phi, phip = sol.sol(N)

    rho_phi, p_phi, K, VV, H2 = scalar_energy_pressure(a, phi, phip, L4)
    rho_m = rho_m_of_a(a)
    rho_r = rho_r_of_a(a)
    rho_tot = rho_m + rho_r + rho_phi
    p_tot = rho_r / 3.0 + p_phi

    return {
        "z": float(z),
        "a": float(a),
        "phi": float(phi),
        "phip": float(phip),
        "H": float(np.sqrt(H2)),
        "H_sq": float(H2),
        "rho_m": float(rho_m),
        "rho_r": float(rho_r),
        "rho_phi": float(rho_phi),
        "p_phi": float(p_phi),
        "rho_tot": float(rho_tot),
        "p_tot": float(p_tot),
        "K": float(K),
        "V": float(VV),
        "w_phi": float(p_phi / rho_phi),
        "w_tot": float(p_tot / rho_tot),
        "q": float(0.5 * (1.0 + 3.0 * p_tot / rho_tot)),
        "Omega_m": float(rho_m / rho_tot),
        "Omega_r": float(rho_r / rho_tot),
        "Omega_phi": float(rho_phi / rho_tot),
        "closure_error": float(rho_tot - 3.0 * H2),
    }

def H_of_z(z, sol, L4):
    return state_at_z(z, sol, L4)["H"]

def q_of_z(z, sol, L4):
    return state_at_z(z, sol, L4)["q"]

def wphi_of_z(z, sol, L4):
    return state_at_z(z, sol, L4)["w_phi"]


# ============================================================
# SECTION H — LCDM REFERENCE
# ============================================================

def H_sq_lcdm(a):
    rho_m = rho_m_of_a(a)
    rho_r = rho_r_of_a(a)
    rho_L = 3.0 * Omega_L
    return (rho_m + rho_r + rho_L) / 3.0

def H_lcdm_of_z(z):
    a = 1.0 / (1.0 + z)
    return np.sqrt(H_sq_lcdm(a))

def state_lcdm_at_z(z):
    a = 1.0 / (1.0 + z)
    H2 = H_sq_lcdm(a)

    rho_m = rho_m_of_a(a)
    rho_r = rho_r_of_a(a)
    rho_L = 3.0 * Omega_L
    rho_tot = rho_m + rho_r + rho_L
    p_tot = rho_r / 3.0 - rho_L

    return {
        "z": float(z),
        "a": float(a),
        "H": float(np.sqrt(H2)),
        "q": float(0.5 * (1.0 + 3.0 * p_tot / rho_tot)),
        "w_tot": float(p_tot / rho_tot)
    }

def delta_q(z_array, sol, L4):
    z_array = np.asarray(z_array)
    out = np.empty_like(z_array, dtype=float)
    for i, z in enumerate(z_array):
        out[i] = q_of_z(z, sol, L4) - state_lcdm_at_z(z)["q"]
    return out

def delta_H_over_H_lcdm(z_array, sol, L4):
    z_array = np.asarray(z_array)
    out = np.empty_like(z_array, dtype=float)
    for i, z in enumerate(z_array):
        Ht = H_of_z(z, sol, L4)
        Hl = H_lcdm_of_z(z)
        out[i] = (Ht - Hl) / Hl
    return out


# ============================================================
# SECTION I — DISTANCES FOR BAO
# ============================================================

def comoving_distance(z, sol, L4):
    # Flat universe: D_M = \int_0^z dz'/H(z')
    val, err = quad(lambda zp: 1.0 / H_of_z(zp, sol, L4), 0.0, z,
                    epsabs=1e-9, epsrel=1e-9, limit=300)
    return val

def DM_over_rd(z, sol, L4, rd_dimless=rd_dimless_default):
    return comoving_distance(z, sol, L4) / rd_dimless

def DH_over_rd(z, sol, L4, rd_dimless=rd_dimless_default):
    return (1.0 / H_of_z(z, sol, L4)) / rd_dimless

def DV_over_rd(z, sol, L4, rd_dimless=rd_dimless_default):
    DM = comoving_distance(z, sol, L4)
    DH = 1.0 / H_of_z(z, sol, L4)
    DV = (z * DM**2 * DH)**(1.0 / 3.0)
    return DV / rd_dimless


# ============================================================
# SECTION J — DESI DR1 BAO DATA VECTOR
# Order:
# 1.  BGS z=0.295: DV/rd
# 2-3. LRG1 z=0.510: DM/rd, DH/rd
# 4-5. LRG2 z=0.706: DM/rd, DH/rd
# 6-7. LRG3+ELG1 z=0.934: DM/rd, DH/rd
# 8-9. ELG2 z=1.317: DM/rd, DH/rd
# 10. QSO z=1.491: DV/rd
# 11-12. Lya z=2.330: DM/rd, DH/rd
# ============================================================

data_vector = np.array([
    7.93,
    13.62, 20.98,
    16.85, 20.08,
    21.71, 17.88,
    27.79, 13.82,
    26.07,
    39.71, 8.52
], dtype=float)

sigma = np.array([
    0.15,
    0.25, 0.61,
    0.32, 0.60,
    0.28, 0.35,
    0.69, 0.42,
    0.67,
    0.94, 0.17
], dtype=float)

C = np.array([
    [0.0225, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0.0625, -0.445*0.25*0.61, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, -0.445*0.25*0.61, 0.3721, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0.1024, -0.420*0.32*0.60, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, -0.420*0.32*0.60, 0.36, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0.0784, -0.389*0.28*0.35, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, -0.389*0.28*0.35, 0.1225, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0.4761, -0.444*0.69*0.42, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, -0.444*0.69*0.42, 0.1764, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.4489, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.8836, -0.477*0.94*0.17],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -0.477*0.94*0.17, 0.0289]
], dtype=float)

Cinv = np.linalg.inv(C)

def calculate_chi_squared(d, m, Cinv):
    diff = np.asarray(d) - np.asarray(m)
    return float(diff.T @ Cinv @ diff)


# ============================================================
# SECTION K — MODEL PREDICTION VECTOR IN DESI ORDER
# ============================================================

def bao_model_vector(sol, L4, rd_dimless=rd_dimless_default):
    return np.array([
        DV_over_rd(0.295, sol, L4, rd_dimless),

        DM_over_rd(0.510, sol, L4, rd_dimless),
        DH_over_rd(0.510, sol, L4, rd_dimless),

        DM_over_rd(0.706, sol, L4, rd_dimless),
        DH_over_rd(0.706, sol, L4, rd_dimless),

        DM_over_rd(0.934, sol, L4, rd_dimless),
        DH_over_rd(0.934, sol, L4, rd_dimless),

        DM_over_rd(1.317, sol, L4, rd_dimless),
        DH_over_rd(1.317, sol, L4, rd_dimless),

        DV_over_rd(1.491, sol, L4, rd_dimless),

        DM_over_rd(2.330, sol, L4, rd_dimless),
        DH_over_rd(2.330, sol, L4, rd_dimless),
    ], dtype=float)

def chi2_tifa(phi_ini, rd_dimless=rd_dimless_default, z_ini=z_ini_default):
    L4, sol, diag = shoot_L4_for_H0_eq_1(phi_ini, z_ini=z_ini)
    m = bao_model_vector(sol, L4, rd_dimless=rd_dimless)
    chi2 = calculate_chi_squared(data_vector, m, Cinv)
    return {
        "phi_ini": phi_ini,
        "phi_ini_over_pi": phi_ini / np.pi,
        "L4": L4,
        "sol": sol,
        "diag0": diag,
        "model_vector": m,
        "chi2": chi2
    }


# ============================================================
# SECTION L — OPTIONAL SCAN
# ============================================================

def scan_phi_ini(phi_over_pi_grid,
                 rd_dimless=rd_dimless_default,
                 z_ini=z_ini_default,
                 verbose=True):
    results = []

    for x in phi_over_pi_grid:
        phi_ini = x * np.pi
        try:
            out = chi2_tifa(phi_ini, rd_dimless=rd_dimless, z_ini=z_ini)
            results.append(out)
            if verbose:
                print(
                    f"phi_ini/pi={x:.4f} | "
                    f"chi2={out['chi2']:.6f} | "
                    f"L4={out['L4']:.8f} | "
                    f"phi0/pi={out['diag0']['phi0']/np.pi:.6f} | "
                    f"w0={out['diag0']['wphi0']:.6f} | "
                    f"q0={out['diag0']['q0']:.6f}"
                )
        except Exception as e:
            if verbose:
                print(f"phi_ini/pi={x:.4f} failed: {e}")

    return results

def best_result(results):
    if len(results) == 0:
        return None
    return min(results, key=lambda r: r["chi2"])


# ============================================================
# SECTION M — EXAMPLE RUN
# ============================================================

if __name__ == "__main__":
    # Example single run
    phi_ini_test = 0.34 * np.pi
    out = chi2_tifa(phi_ini_test, rd_dimless=rd_dimless_default)

    print("\n=== SINGLE RUN ===")
    print(f"phi_ini/pi          = {out['phi_ini_over_pi']:.8f}")
    print(f"L4                  = {out['L4']:.12g}")
    print(f"chi2                = {out['chi2']:.8f}")
    print(f"phi0/pi             = {out['diag0']['phi0']/np.pi:.8f}")
    print(f"wphi0               = {out['diag0']['wphi0']:.8f}")
    print(f"q0                  = {out['diag0']['q0']:.8f}")
    print(f"H0                  = {out['diag0']['H0']:.8f}")
    print(f"closure error       = {out['diag0']['closure_error']:.3e}")

    # Example scan
    grid = np.linspace(0.10, 0.90, 17)
    results = scan_phi_ini(grid, rd_dimless=rd_dimless_default, verbose=True)

    best = best_result(results)
    if best is not None:
        print("\n=== BEST SCAN RESULT ===")
        print(f"best phi_ini/pi     = {best['phi_ini_over_pi']:.8f}")
        print(f"best chi2           = {best['chi2']:.8f}")
        print(f"best L4             = {best['L4']:.12g}")
        print(f"best phi0/pi        = {best['diag0']['phi0']/np.pi:.8f}")
        print(f"best wphi0          = {best['diag0']['wphi0']:.8f}")
        print(f"best q0             = {best['diag0']['q0']:.8f}")

/tmp/ipykernel_218/4119279723.py:342: IntegrationWarning: The occurrence of roundoff error is detected, which prevents 
  the requested tolerance from being achieved.  The error may be 
  underestimated.
  val, err = quad(lambda zp: 1.0 / H_of_z(zp, sol, L4), 0.0, z,



=== SINGLE RUN ===
phi_ini/pi          = 0.34000000
L4                  = 5146066560.86
chi2                = 3548.01313913
phi0/pi             = 0.00001442
wphi0               = -0.28445411
q0                  = 0.20780690
H0                  = 0.99999964
closure error       = 0.000e+00


In [ ]:

import numpy as np
from scipy.integrate import solve_ivp, quad
import matplotlib.pyplot as plt

# ── Parameters ─────────────────────────────────────────────
H0_kmsMpc = 67.36
Omega_m   = 0.3153
Omega_r   = 9.0e-5
Omega_L   = 1.0 - Omega_m - Omega_r
f_E       = 2.0
rd        = 147.09
c_kms     = 2.998e5

# ── DESI data ───────────────────────────────────────────────
data_vector = np.array([
    7.93,
    13.62, 20.98,
    16.85, 20.08,
    21.71, 17.88,
    27.79, 13.82,
    26.07,
    39.71, 8.52
])

C = np.array([
    [0.0225, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0.0625, -0.06786, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, -0.06786, 0.3721, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0.1024, -0.08064, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, -0.08064, 0.36, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0.0784, -0.03812, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, -0.03812, 0.1225, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0.4761, -0.12867, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, -0.12867, 0.1764, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0.4489, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.8836, -0.07622],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -0.07622, 0.0289]
], dtype=float)

Cinv = np.linalg.inv(C)

# ── Global Lambda4 ──────────────────────────────────────────
Lambda4_global = Omega_L

def V(phi):
    return Lambda4_global * (1.0 - np.cos(phi / f_E))

def dV(phi):
    return (Lambda4_global / f_E) * np.sin(phi / f_E)

def H_sq(a, phi, dphi):
    rho_m = Omega_m / a**3
    rho_r = Omega_r / a**4
    num = rho_m + rho_r + V(phi)
    den = max(1.0 - 0.5 * dphi**2, 0.1)
    return num / den

def equations(N, state):
    phi, dphi = state
    a   = np.exp(N)
    H2  = H_sq(a, phi, dphi)
    H   = np.sqrt(max(H2, 1e-30))
    rm  = Omega_m / a**3
    rr  = Omega_r / a**4
    dH  = -0.5*(rm + (4.0/3.0)*rr + H2*dphi**2) / H
    ddp = -(dH/H + 3.0)*dphi - dV(phi)/H2
    return [dphi, ddp]

def run_integration(phi_ini, L4, z_ini=500):
    global Lambda4_global
    Lambda4_global = L4
    N_ini = np.log(1.0/(1.0 + z_ini))
    sol = solve_ivp(
        equations, [N_ini, 0.0], [phi_ini, 0.0],
        method='RK45', rtol=1e-8, atol=1e-10,
        max_step=0.01, dense_output=True
    )
    return sol

def find_Lambda4_thawing(phi_ini):
    """
    Find Lambda4 for thawing branch only.
    Key constraint: phi_today must stay near phi_ini.
    Field displacement < 20% of phi_ini.
    """
    global Lambda4_global

    # Thawing branch: Lambda4 must be small.
    # Estimate: V(phi_ini) ~ Omega_L
    # So Lambda4 ~ Omega_L / (1 - cos(phi_ini/f_E))
    x = phi_ini / f_E
    if abs(1.0 - np.cos(x)) < 1e-10:
        return None, None
    L4 = Omega_L / (1.0 - np.cos(x))

    # Hard cap — thawing branch never needs huge Lambda4
    L4_max = 10.0 * Omega_L
    if L4 > L4_max:
        return None, None

    for iteration in range(20):
        sol = run_integration(phi_ini, L4)
        if not sol.success:
            return None, None

        phi_t = sol.y[0, -1]
        dph_t = sol.y[1, -1]

        # Thawing branch check:
        # field should not have rolled far
        if phi_ini > 1e-6:
            displacement = abs(phi_t - phi_ini) / abs(phi_ini)
            if displacement > 0.5:
                # Wrong branch — field rolled too far
                return None, None

        Lambda4_global = L4
        H2_t  = H_sq(1.0, phi_t, dph_t)
        K_t   = 0.5 * H2_t * dph_t**2
        rho   = K_t + V(phi_t)

        if abs(rho - Omega_L) / Omega_L < 1e-4:
            break

        L4 = L4 * (Omega_L / rho)

        # Re-check cap after rescaling
        if L4 > L4_max:
            return None, None

    # Final viability check
    Lambda4_global = L4
    phi_t = sol.y[0, -1]
    dph_t = sol.y[1, -1]
    H2_t  = H_sq(1.0, phi_t, dph_t)
    K_t   = 0.5 * H2_t * dph_t**2
    rho   = K_t + V(phi_t)
    if rho < 1e-6:
        return None, None
    w0 = (K_t - V(phi_t)) / rho
    if w0 > -0.5:
        # Not dark energy like — reject
        return None, None

    return L4, sol

def H_at_z(z, sol, L4):
    global Lambda4_global
    Lambda4_global = L4
    N = np.log(1.0/(1.0 + z))
    phi, dphi = sol.sol(N)
    a = 1.0/(1.0 + z)
    return np.sqrt(max(H_sq(a, phi, dphi), 1e-30))

def DM_at_z(z, sol, L4):
    def integrand(zp):
        return c_kms / (H_at_z(zp, sol, L4) * H0_kmsMpc)
    val, _ = quad(integrand, 0.0, z,
                  limit=200, epsabs=1e-6, epsrel=1e-6)
    return val

def bao_vector(sol, L4):
    m = []
    # BGS z=0.295 isotropic
    DM = DM_at_z(0.295, sol, L4)
    DH = c_kms / (H_at_z(0.295, sol, L4) * H0_kmsMpc)
    m.append((0.295 * DM**2 * DH)**(1.0/3.0) / rd)
    # LRG1 z=0.510
    DM = DM_at_z(0.510, sol, L4)
    DH = c_kms / (H_at_z(0.510, sol, L4) * H0_kmsMpc)
    m.append(DM / rd)
    m.append(DH / rd)
    # LRG2 z=0.706
    DM = DM_at_z(0.706, sol, L4)
    DH = c_kms / (H_at_z(0.706, sol, L4) * H0_kmsMpc)
    m.append(DM / rd)
    m.append(DH / rd)
    # LRG3+ELG1 z=0.934
    DM = DM_at_z(0.934, sol, L4)
    DH = c_kms / (H_at_z(0.934, sol, L4) * H0_kmsMpc)
    m.append(DM / rd)
    m.append(DH / rd)
    # ELG2 z=1.317
    DM = DM_at_z(1.317, sol, L4)
    DH = c_kms / (H_at_z(1.317, sol, L4) * H0_kmsMpc)
    m.append(DM / rd)
    m.append(DH / rd)
    # QSO z=1.491 isotropic
    DM = DM_at_z(1.491, sol, L4)
    DH = c_kms / (H_at_z(1.491, sol, L4) * H0_kmsMpc)
    m.append((1.491 * DM**2 * DH)**(1.0/3.0) / rd)
    # Lya z=2.330
    DM = DM_at_z(2.330, sol, L4)
    DH = c_kms / (H_at_z(2.330, sol, L4) * H0_kmsMpc)
    m.append(DM / rd)
    m.append(DH / rd)
    return m

# ── Scan over phi_today directly ────────────────────────────
# Scan phi_ini and record phi_today
# Focus on thawing branch: small phi_ini
phi_scan = np.linspace(0.05, 0.55, 25) * np.pi * f_E

results = []

for phi_ini in phi_scan:
    L4, sol = find_Lambda4_thawing(phi_ini)
    if L4 is None:
        ini_norm = phi_ini / (np.pi * f_E)
        print(f"phi_ini/pi={ini_norm:.3f} | REJECTED (wrong branch)")
        continue

    global Lambda4_global
    Lambda4_global = L4
    phi_t = sol.y[0, -1]
    dph_t = sol.y[1, -1]
    H2_t  = H_sq(1.0, phi_t, dph_t)
    K_t   = 0.5 * H2_t * dph_t**2
    rho   = K_t + V(phi_t)
    w0    = (K_t - V(phi_t)) / rho

    try:
        bao  = bao_vector(sol, L4)
        diff = data_vector - np.array(bao)
        chi2 = float(diff.T @ Cinv @ diff)
    except Exception as e:
        print(f"BAO failed: {e}")
        continue

    coord = phi_t / (np.pi * f_E)
    ini_n = phi_ini / (np.pi * f_E)
    results.append((coord, w0, chi2, L4))
    print(f"phi_ini/pi={ini_n:.3f} | phi_today/pi={coord:.3f} | "
          f"w0={w0:.4f} | chi2={chi2:.2f} | L4={L4:.4f}")

# ── Plot ────────────────────────────────────────────────────
if results:
    coords, w0s, chi2s, _ = zip(*results)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8))

    ax1.plot(coords, chi2s, 'b-o', ms=5)
    ax1.axhline(12.66, color='g', ls='--',
                label='DESI LCDM 12.66')
    ax1.axhline(13.54, color='r', ls='--',
                label='Paper A 13.54')
    ax1.axvline(0.340, color='orange', ls=':',
                label='phi/pi=0.340')
    ax1.set_xlabel('phi_today / pi')
    ax1.set_ylabel('chi2')
    ax1.set_title('Consistency Test (thawing branch only)')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(coords, w0s, 'r-o', ms=5)
    ax2.axhline(-0.899, color='b', ls='--',
                label='DESI w0=-0.899')
    ax2.axhline(-1.0, color='k', ls=':',
                label='LCDM w0=-1')
    ax2.axvline(0.340, color='orange', ls=':',
                label='phi/pi=0.340')
    ax2.set_xlabel('phi_today / pi')
    ax2.set_ylabel('w0')
    ax2.set_title('Equation of state (thawing branch)')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

    best = min(results, key=lambda x: x[2])
    print(f"\n── RESULT ──────────────────────────────")
    print(f"Best fit phi_today/pi = {best[0]:.4f}")
    print(f"         w0           = {best[1]:.4f}")
    print(f"         chi2         = {best[2]:.2f}")
    print(f"         Lambda4      = {best[3]:.4f}")
    print(f"Paper A: phi/pi=0.340, chi2=13.54")
    gap = abs(best[0] - 0.340)
    print(f"Gap: {gap:.4f} in phi/pi")
    if gap < 0.05:
        print("CONSISTENCY: PASSED")
    elif gap < 0.15:
        print("CONSISTENCY: MARGINAL")
    else:
        print("CONSISTENCY: FAILED")

In [ ]:

# Refine around minimum
phi_refined = np.linspace(0.17, 0.23, 20) * np.pi * f_E

results_refined = []
for phi_ini in phi_refined:
    L4, sol = find_Lambda4_thawing(phi_ini)
    if L4 is None:
        continue
    Lambda4_global = L4
    phi_t = sol.y[0, -1]
    dph_t = sol.y[1, -1]
    H2_t  = H_sq(1.0, phi_t, dph_t)
    K_t   = 0.5 * H2_t * dph_t**2
    rho   = K_t + V(phi_t)
    w0    = (K_t - V(phi_t)) / rho
    bao   = bao_vector(sol, L4)
    diff  = data_vector - np.array(bao)
    chi2  = float(diff.T @ Cinv @ diff)
    coord = phi_t / (np.pi * f_E)
    results_refined.append((coord, w0, chi2, L4))
    print(f"phi/pi={coord:.4f} | w0={w0:.5f} | chi2={chi2:.3f}")

best_r = min(results_refined, key=lambda x: x[2])
print(f"\nRefined minimum:")
print(f"phi/pi = {best_r[0]:.4f}")
print(f"w0     = {best_r[1]:.5f}")
print(f"chi2   = {best_r[2]:.3f}")

In [ ]:

# Test whether f_E = 1.0 shifts minimum to 0.340
f_E = 1.0

# Re-run the full scan with f_E = 1.0
# Everything else identical
# If minimum shifts to phi/pi ~ 0.340
# then Paper A used f_E = M_Pl not 2M_Pl

In [ ]:

# ── f_E sensitivity test ────────────────────────────────────
# Test whether f_E = 1.0 shifts minimum toward phi/pi = 0.340

f_E = 1.0  # Change from 2.0 to 1.0

results_fE1 = []
phi_scan_fE1 = np.linspace(0.10, 0.55, 25) * np.pi * f_E

for phi_ini in phi_scan_fE1:
    L4, sol = find_Lambda4_thawing(phi_ini)
    if L4 is None:
        continue
    Lambda4_global = L4
    phi_t = sol.y[0, -1]
    dph_t = sol.y[1, -1]
    H2_t  = H_sq(1.0, phi_t, dph_t)
    K_t   = 0.5 * H2_t * dph_t**2
    rho   = K_t + V(phi_t)
    w0    = (K_t - V(phi_t)) / rho
    try:
        bao  = bao_vector(sol, L4)
        diff = data_vector - np.array(bao)
        chi2 = float(diff.T @ Cinv @ diff)
    except:
        continue
    coord = phi_t / (np.pi * f_E)
    results_fE1.append((coord, w0, chi2))
    print(f"phi/pi={coord:.4f} | w0={w0:.5f} | chi2={chi2:.3f}")

if results_fE1:
    best = min(results_fE1, key=lambda x: x[2])
    print(f"\nf_E=1.0 minimum:")
    print(f"phi/pi = {best[0]:.4f}")
    print(f"w0     = {best[1]:.5f}")
    print(f"chi2   = {best[2]:.3f}")

# Reset f_E back to 2.0 after test
f_E = 2.0

In [ ]:

# f_E = 1.0 refined scan
f_E = 1.0

phi_refined2 = np.linspace(0.30, 0.42, 25) * np.pi * f_E
results_refined2 = []

for phi_ini in phi_refined2:
    L4, sol = find_Lambda4_thawing(phi_ini)
    if L4 is None:
        continue
    Lambda4_global = L4
    phi_t = sol.y[0, -1]
    dph_t = sol.y[1, -1]
    H2_t  = H_sq(1.0, phi_t, dph_t)
    K_t   = 0.5 * H2_t * dph_t**2
    rho   = K_t + V(phi_t)
    w0    = (K_t - V(phi_t)) / rho
    try:
        bao  = bao_vector(sol, L4)
        diff = data_vector - np.array(bao)
        chi2 = float(diff.T @ Cinv @ diff)
    except:
        continue
    coord = phi_t / (np.pi * f_E)
    results_refined2.append((coord, w0, chi2))
    print(f"phi/pi={coord:.4f} | w0={w0:.5f} | chi2={chi2:.4f}")

if results_refined2:
    best2 = min(results_refined2, key=lambda x: x[2])
    print(f"\nf_E=1.0 refined minimum:")
    print(f"phi/pi = {best2[0]:.4f}")
    print(f"w0     = {best2[1]:.5f}")
    print(f"chi2   = {best2[2]:.4f}")
    print(f"Paper A phi/pi = 0.340")
    print(f"Gap = {abs(best2[0]-0.340):.4f}")

f_E = 2.0  # reset

In [ ]:

f_E = 1.0

coords = [r[0] for r in results_refined2]
chi2s  = [r[2] for r in results_refined2]
w0s    = [r[1] for r in results_refined2]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8,8))

ax1.plot(coords, chi2s, 'b-o', ms=4)
ax1.axhline(13.54, color='r', ls='--',
            label='Paper A chi2=13.54')
ax1.axhline(12.66, color='g', ls='--',
            label='DESI LCDM chi2=12.66')
ax1.axvline(0.340, color='orange', ls=':',
            label='Paper A phi/pi=0.340')
ax1.axvline(0.3491, color='purple', ls=':',
            label='Dynamical min=0.349')
ax1.set_xlabel('phi_today / pi  (f_E = M_Pl)')
ax1.set_ylabel('chi2')
ax1.set_title('Consistency Test PASSED: Gap = 0.009')
ax1.set_ylim(12, 20)
ax1.legend()
ax1.grid(True)

ax2.plot(coords, w0s, 'r-o', ms=4)
ax2.axhline(-0.899, color='b', ls='--',
            label='DESI w0=-0.899')
ax2.axhline(-0.9044, color='purple', ls=':',
            label='Dynamical w0=-0.904')
ax2.axvline(0.340, color='orange', ls=':',
            label='Paper A phi/pi=0.340')
ax2.set_xlabel('phi_today / pi  (f_E = M_Pl)')
ax2.set_ylabel('w0')
ax2.set_title('Equation of State: w0 agreement')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('consistency_test_PASSED.png', dpi=150)
plt.show()

print("Gap in phi/pi:", abs(0.3491 - 0.340))
print("Gap in w0:    ", abs(-0.9044 - (-0.899)))
print("Gap in chi2:  ", abs(13.4119 - 13.54))
print("CONSISTENCY TEST: PASSED")

f_E = 2.0

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Data from refined scan
coords = [0.2369, 0.2437, 0.2505, 0.2571,
          0.2637, 0.2701, 0.2765, 0.2828,
          0.2891, 0.2953, 0.3014, 0.3075,
          0.3136, 0.3196, 0.3256, 0.3315,
          0.3374, 0.3433, 0.3491, 0.3549,
          0.3607, 0.3665, 0.3722, 0.3779,
          0.3836]

chi2s = [28.021, 25.314, 23.084, 21.225,
         19.705, 18.447, 17.405, 16.545,
         15.836, 15.257, 14.779, 14.402,
         14.103, 13.870, 13.695, 13.567,
         13.481, 13.431, 13.412, 13.419,
         13.448, 13.496, 13.564, 13.643,
         13.733]

coords = np.array(coords)
chi2s  = np.array(chi2s)

chi2_min  = chi2s.min()
phi_min   = coords[chi2s.argmin()]
threshold = chi2_min + 1.0  # 1-sigma

# Find crossing points by interpolation
from scipy.interpolate import interp1d

f_interp = interp1d(coords, chi2s, kind='cubic')
phi_fine  = np.linspace(coords[0], coords[-1], 5000)
chi2_fine = f_interp(phi_fine)

# Left crossing
left_mask  = (phi_fine < phi_min) & (chi2_fine < threshold)
right_mask = (phi_fine > phi_min) & (chi2_fine < threshold)

phi_left  = phi_fine[left_mask].min()  if left_mask.any()  else None
phi_right = phi_fine[right_mask].max() if right_mask.any() else None

sigma_width = (phi_right - phi_left) / 2.0
gap         = abs(phi_min - 0.340)
gap_sigma   = gap / sigma_width

print(f"chi2 minimum:      {chi2_min:.4f}")
print(f"phi/pi at minimum: {phi_min:.4f}")
print(f"1-sigma threshold: {threshold:.4f}")
print(f"1-sigma left edge: {phi_left:.4f}")
print(f"1-sigma right edge:{phi_right:.4f}")
print(f"1-sigma half-width:{sigma_width:.4f}")
print(f"Paper A gap:        {gap:.4f}")
print(f"Gap in sigma:       {gap_sigma:.3f} sigma")

# Plot
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(phi_fine, chi2_fine, 'b-', lw=2,
        label='chi2 (cubic interp)')
ax.scatter(coords, chi2s, color='blue',
           s=20, zorder=5)
ax.axhline(chi2_min, color='purple',
           ls=':', label=f'min={chi2_min:.3f}')
ax.axhline(threshold, color='gray',
           ls='--', label=f'1-sigma={threshold:.3f}')
ax.axhline(13.54, color='red',
           ls='--', label='Paper A=13.54')
ax.axhline(12.66, color='green',
           ls='--', label='LCDM=12.66')
ax.axvline(0.340, color='orange',
           ls=':', label='Paper A phi=0.340')
ax.axvline(phi_min, color='purple',
           ls=':', label=f'Min phi={phi_min:.3f}')
ax.fill_betweenx([12, threshold],
                  phi_left, phi_right,
                  alpha=0.15, color='blue',
                  label='1-sigma region')
ax.set_xlim(0.28, 0.42)
ax.set_ylim(12, 18)
ax.set_xlabel('phi_today / pi  (f_E = M_Pl)')
ax.set_ylabel('chi2')
ax.set_title(
    f'1-sigma analysis: Paper A at {gap_sigma:.2f}σ from minimum')
ax.legend(fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.savefig('sigma_analysis.png', dpi=150)
plt.show()

In [ ]:

# Plot phi(N) trajectory at best fit
f_E = 1.0
phi_best = 0.3491 * np.pi * f_E

L4, sol = find_Lambda4_thawing(phi_best)

N_arr   = np.linspace(np.log(1/501), 0, 500)
phi_arr = []
w0_arr  = []
z_arr   = []

Lambda4_global = L4
for N in N_arr:
    a = np.exp(N)
    z = 1.0/a - 1.0
    ph, dph = sol.sol(N)
    H2 = H_sq(a, ph, dph)
    K  = 0.5 * H2 * dph**2
    rho_phi = K + V(ph)
    if rho_phi > 1e-10:
        w = (K - V(ph)) / rho_phi
    else:
        w = -1.0
    phi_arr.append(ph / (np.pi * f_E))
    w0_arr.append(w)
    z_arr.append(z)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8,8))

ax1.plot(z_arr, phi_arr, 'b-', lw=2)
ax1.axvline(0, color='k', ls=':', alpha=0.5)
ax1.axhline(0.349, color='purple', ls='--',
            label='phi/pi today = 0.349')
ax1.axhline(0.340, color='orange', ls='--',
            label='Paper A = 0.340')
ax1.set_xscale('log')
ax1.set_xlabel('redshift z')
ax1.set_ylabel('phi(z) / pi')
ax1.set_title('Field trajectory: thawing branch')
ax1.legend()
ax1.grid(True)
ax1.invert_xaxis()

ax2.plot(z_arr, w0_arr, 'r-', lw=2)
ax2.axhline(-0.899, color='b', ls='--',
            label='DESI w0=-0.899')
ax2.axhline(-1.0, color='k', ls=':',
            label='LCDM w0=-1')
ax2.axvline(0, color='k', ls=':', alpha=0.5)
ax2.set_xscale('log')
ax2.set_xlabel('redshift z')
ax2.set_ylabel('w(z)')
ax2.set_title('Equation of state evolution')
ax2.set_ylim(-1.05, -0.5)
ax2.legend()
ax2.grid(True)
ax2.invert_xaxis()

# Fix both axes
ax1.set_xlim(500, 0.005)
ax2.set_xlim(500, 0.005)
# Remove ax1.invert_xaxis()
# Remove ax2.invert_xaxis()

plt.tight_layout()
plt.savefig('field_trajectory.png', dpi=150)
plt.show()

In [ ]:

# Use phi_ini that gives phi_today = 0.349
# From the refined scan the best fit was:
# phi_ini/pi = 0.3491 INPUT
# which gives phi_today/pi = 0.3491 OUTPUT?

# Actually re-check:
# In find_Lambda4_thawing(phi_ini)
# phi_ini IS the initial value at high z
# and phi_today is sol.y[0,-1]

# Print both for the best fit point:
f_E = 1.0
phi_ini_best = 0.3491 * np.pi * f_E
L4, sol = find_Lambda4_thawing(phi_ini_best)
phi_today_check = sol.y[0, -1] / (np.pi * f_E)
print(f"phi_ini/pi  = {0.3491:.4f}")
print(f"phi_today/pi = {phi_today_check:.4f}")
# These two numbers will tell us
# what is actually happening

In [ ]:

f_E = 1.0
phi_ini_best = 0.3491 * np.pi * f_E
L4, sol = find_Lambda4_thawing(phi_ini_best)

print("sol.y[0, 0]  =", sol.y[0,  0] / (np.pi * f_E),
      "← index 0")
print("sol.y[0,-1]  =", sol.y[0, -1] / (np.pi * f_E),
      "← index -1")
print("sol.t[ 0]    =", sol.t[ 0],
      "← N at index 0")
print("sol.t[-1]    =", sol.t[-1],
      "← N at index -1")
print("N_today      =", 0.0)
print("N_ini        =", np.log(1/501))

In [ ]:

f_E = 1.0
results_fixed = []

phi_scan = np.linspace(0.30, 0.55, 30) * np.pi * f_E

for phi_ini in phi_scan:
    L4, sol = find_Lambda4_thawing(phi_ini)
    if L4 is None:
        continue
    Lambda4_global = L4

    # CORRECT: use index -1 for today
    phi_today = sol.y[0, -1]          # N=0 is index -1
    dph_today = sol.y[1, -1]

    H2_t = H_sq(1.0, phi_today, dph_today)
    K_t  = 0.5 * H2_t * dph_today**2
    rho  = K_t + V(phi_today)
    w0   = (K_t - V(phi_today)) / rho

    try:
        bao  = bao_vector(sol, L4)
        diff = data_vector - np.array(bao)
        chi2 = float(diff.T @ Cinv @ diff)
    except:
        continue

    # CORRECT coordinate label
    coord_today = phi_today / (np.pi * f_E)
    coord_ini   = phi_ini   / (np.pi * f_E)

    results_fixed.append((coord_ini, coord_today, w0, chi2, L4))
    print(f"phi_ini/pi={coord_ini:.4f} | "
          f"phi_today/pi={coord_today:.4f} | "
          f"w0={w0:.5f} | chi2={chi2:.4f}")

if results_fixed:
    best = min(results_fixed, key=lambda x: x[3])
    print(f"\nCORRECTED minimum:")
    print(f"phi_ini/pi   = {best[0]:.4f}")
    print(f"phi_today/pi = {best[1]:.4f}")
    print(f"w0           = {best[2]:.5f}")
    print(f"chi2         = {best[3]:.4f}")
    print(f"Paper A      = 0.340")
    print(f"Gap          = {abs(best[1]-0.340):.4f}")

In [ ]:

f_E = 1.0
results_final = []

# Scan around minimum
phi_scan2 = np.linspace(0.37, 0.42, 25) * np.pi * f_E

for phi_ini in phi_scan2:
    L4, sol = find_Lambda4_thawing(phi_ini)
    if L4 is None:
        continue
    Lambda4_global = L4

    phi_today = sol.y[0, -1]
    dph_today = sol.y[1, -1]

    H2_t = H_sq(1.0, phi_today, dph_today)
    K_t  = 0.5 * H2_t * dph_today**2
    rho  = K_t + V(phi_today)
    w0   = (K_t - V(phi_today)) / rho

    try:
        bao  = bao_vector(sol, L4)
        diff = data_vector - np.array(bao)
        chi2 = float(diff.T @ Cinv @ diff)
    except:
        continue

    coord_today = phi_today / (np.pi * f_E)
    coord_ini   = phi_ini   / (np.pi * f_E)

    results_final.append((coord_ini, coord_today, w0, chi2))
    print(f"phi_ini/pi={coord_ini:.4f} | "
          f"phi_today/pi={coord_today:.4f} | "
          f"w0={w0:.5f} | chi2={chi2:.4f}")

if results_final:
    best = min(results_final, key=lambda x: x[3])
    print(f"\nFINAL minimum:")
    print(f"phi_ini/pi   = {best[0]:.4f}")
    print(f"phi_today/pi = {best[1]:.4f}")
    print(f"w0           = {best[2]:.5f}")
    print(f"chi2         = {best[3]:.4f}")
    print(f"Paper A      = 0.340")
    print(f"Gap          = {abs(best[1]-0.340):.4f}")

In [ ]:

import numpy as np

# Best fit values from corrected scan
f_E       = 1.0                        # in M_Pl units
phi_ini   = 0.3908 * np.pi * f_E       # best fit phi_ini

# Get Lambda4 at best fit
L4, sol = find_Lambda4_thawing(phi_ini)
print(f"Lambda^4 (code units, M_Pl=1): {L4:.6e}")

# Mass of scalar field
# V(phi) = Lambda^4 * (1 - cos(phi/f_E))
# V''(phi) = Lambda^4 / f_E^2 * cos(phi/f_E)
V_pp     = L4 / f_E**2 * np.cos(phi_ini / f_E)
m_phi_Pl = np.sqrt(abs(V_pp))          # in M_Pl units
print(f"m_phi (M_Pl units):             {m_phi_Pl:.6e}")

# Convert to eV
# Reduced Planck mass M_Pl = 2.435e27 eV
M_Pl_eV  = 2.435e27                    # eV
m_phi_eV = m_phi_Pl * M_Pl_eV
print(f"m_phi (eV):                     {m_phi_eV:.6e}")
print(f"m_phi (eV):                     {m_phi_eV:.3e}")
print()
print(f"Fuzzy DM target:                ~1e-22 eV")
print(f"Quintessence target (H0):       ~1e-33 eV")
print(f"Ratio to fuzzy DM:              {m_phi_eV / 1e-22:.3e}")
print(f"Ratio to H0 scale:              {m_phi_eV / 1e-33:.3e}")

In [ ]:

# Physical mass from H0 normalisation
# In code: H0^2 = Omega_m + Omega_r + Omega_DE
# = 0.315 + 8.24e-5 + 0.685 = 1.000
# So H0 = 1 in code units.

# Physical H0 in eV:
H0_eV   = 1.4622e-33   # eV  (67.4 km/s/Mpc)

# Code Lambda in physical units:
# Lambda_code^4 = Lambda_phys^4 / H0^4 * (something)
# Need to track how Lambda enters H_sq

# The correct conversion:
# m_phi_code = 0.634 (in units of H0, since H0=1)
# m_phi_phys = 0.634 * H0_phys

m_phi_code = 6.337564e-01   # in H0 units
m_phi_eV_correct = m_phi_code * H0_eV

print(f"m_phi (H0 units):      {m_phi_code:.4f}")
print(f"H0 (eV):               {H0_eV:.4e}")
print(f"m_phi physical (eV):   {m_phi_eV_correct:.4e}")
print(f"Fuzzy DM (~1e-22 eV):  ratio = "
      f"{m_phi_eV_correct/1e-22:.3e}")
print(f"Quintessence (~H0):    ratio = "
      f"{m_phi_eV_correct/H0_eV:.3f}")

In [ ]:

import numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.interpolate import interp1d

# ── 1. Build H(z) from best-fit solution ──────────────────
f_E     = 1.0
phi_ini = 0.3908 * np.pi * f_E
L4, sol = find_Lambda4_thawing(phi_ini)
Lambda4_global = L4

# Fine N grid from ini to today
N_arr  = np.linspace(sol.t[0], 0.0, 2000)
a_arr  = np.exp(N_arr)
z_arr  = 1.0 / a_arr - 1.0

phi_arr  = []
dphi_arr = []
H_arr    = []
wDE_arr  = []

for N in N_arr:
    ph, dph = sol.sol(N)
    H2  = H_sq(np.exp(N), ph, dph)
    H   = np.sqrt(abs(H2))
    K   = 0.5 * H2 * dph**2
    rho = K + V(ph)
    w   = (K - V(ph)) / rho if rho > 1e-15 else -1.0
    phi_arr.append(ph)
    dphi_arr.append(dph)
    H_arr.append(H)
    wDE_arr.append(w)

phi_arr  = np.array(phi_arr)
H_arr    = np.array(H_arr)
wDE_arr  = np.array(wDE_arr)

# Interpolate H(a) and H'(a)
a_sorted  = a_arr[::-1]        # ascending
H_sorted  = H_arr[::-1]
H_interp  = interp1d(a_sorted, H_sorted,
                     kind='cubic', fill_value='extrapolate')
dH_interp = lambda a: np.gradient(
                H_interp(np.linspace(a*0.999, a*1.001, 5)),
                np.linspace(a*0.999, a*1.001, 5))[2]

print("H(z=0) TIFA  :", H_interp(1.0))
print("H(z=0) expect: 1.000 (code units)")

# ── 2. Omega_m(z) ─────────────────────────────────────────
Omega_m0 = 0.315

def Omega_m_z(a):
    H2  = H_interp(a)**2
    return Omega_m0 / (a**3 * H2)

# ── 3. Growth ODE  ────────────────────────────────────────
# Variables: y = [D, dD/da]
# Equation:
# d^2D/da^2 = - (3/a + dH/H/da) dD/da
#             + 3 Omega_m(a) / (2 a^5 H^2) D

def growth_ode(a, y):
    D, Dp = y
    H    = H_interp(a)
    # Numerical derivative of H w.r.t. a
    da   = a * 1e-4
    dH   = (H_interp(a + da) - H_interp(a - da)) / (2 * da)
    Om   = Omega_m_z(a)
    # Coefficients
    coeff_Dp = -(3.0/a + dH/H)
    coeff_D  =  1.5 * Om / (a**2 * H**2)
    return [Dp, coeff_Dp * Dp + coeff_D * D]

# Initial conditions deep in matter era (a_ini = 1/51, z=50)
# Matter dominated: D ~ a, D' = 1
a_ini  = 1.0 / 51.0
D_ini  = a_ini
Dp_ini = 1.0

a_eval = np.linspace(a_ini, 1.0, 2000)

sol_growth_TIFA = solve_ivp(
    growth_ode,
    [a_ini, 1.0],
    [D_ini, Dp_ini],
    method='RK45',
    t_eval=a_eval,
    rtol=1e-8, atol=1e-10,
    dense_output=True
)

# ── 4. LCDM growth for comparison ─────────────────────────
Omega_L = 1.0 - Omega_m0   # flat LCDM, no radiation

def H_LCDM(a):
    return np.sqrt(Omega_m0 / a**3 + Omega_L)

def Omega_m_LCDM(a):
    return Omega_m0 / (a**3 * H_LCDM(a)**2)

def growth_ode_LCDM(a, y):
    D, Dp = y
    H    = H_LCDM(a)
    da   = a * 1e-4
    dH   = (H_LCDM(a+da) - H_LCDM(a-da)) / (2*da)
    Om   = Omega_m_LCDM(a)
    coeff_Dp = -(3.0/a + dH/H)
    coeff_D  =  1.5 * Om / (a**2 * H**2)
    return [Dp, coeff_Dp * Dp + coeff_D * D]

sol_growth_LCDM = solve_ivp(
    growth_ode_LCDM,
    [a_ini, 1.0],
    [D_ini, Dp_ini],
    method='RK45',
    t_eval=a_eval,
    rtol=1e-8, atol=1e-10,
    dense_output=True
)

# ── 5. Results ────────────────────────────────────────────
D_TIFA_today = sol_growth_TIFA.y[0, -1]
D_LCDM_today = sol_growth_LCDM.y[0, -1]

# Normalise both to D(z=0) = 1
D_TIFA_norm = sol_growth_TIFA.y[0] / D_TIFA_today
D_LCDM_norm = sol_growth_LCDM.y[0] / D_LCDM_today

# sigma8 suppression
# Planck LCDM sigma8 = 0.811
sigma8_LCDM  = 0.811
sigma8_TIFA  = sigma8_LCDM * (D_TIFA_today / D_LCDM_today)

# S8
S8_LCDM = sigma8_LCDM * np.sqrt(Omega_m0 / 0.3)
S8_TIFA = sigma8_TIFA  * np.sqrt(Omega_m0 / 0.3)

print(f"\n── Growth factor results ──────────────────")
print(f"D(z=0) TIFA        : {D_TIFA_today:.6f}")
print(f"D(z=0) LCDM        : {D_LCDM_today:.6f}")
print(f"Ratio D_TIFA/D_LCDM: {D_TIFA_today/D_LCDM_today:.6f}")
print(f"")
print(f"sigma8 LCDM (input): {sigma8_LCDM:.4f}")
print(f"sigma8 TIFA        : {sigma8_TIFA:.4f}")
print(f"Delta sigma8       : {sigma8_TIFA - sigma8_LCDM:.4f}")
print(f"")
print(f"S8 LCDM            : {S8_LCDM:.4f}")
print(f"S8 TIFA            : {S8_TIFA:.4f}")
print(f"Delta S8           : {S8_TIFA - S8_LCDM:.4f}")
print(f"")
print(f"KiDS-1000 S8       :  0.766 ± 0.020")
print(f"Planck LCDM S8     :  0.832 ± 0.013")
print(f"TIFA S8            : {S8_TIFA:.3f}")

# ── 6. Plot ───────────────────────────────────────────────
import matplotlib.pyplot as plt

z_plot = 1.0/a_eval - 1.0

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8))

# Growth factor
ax1.plot(z_plot, D_TIFA_norm, 'b-',  lw=2, label='TIFA')
ax1.plot(z_plot, D_LCDM_norm, 'k--', lw=2, label='ΛCDM')
ax1.set_xlabel('redshift z')
ax1.set_ylabel('D(z) / D(0)')
ax1.set_title('Growth factor: TIFA vs ΛCDM')
ax1.legend()
ax1.grid(True)
ax1.set_xlim(0, 3)

# Ratio
ratio = D_TIFA_norm / D_LCDM_norm
ax2.plot(z_plot, ratio, 'r-', lw=2)
ax2.axhline(1.0, color='k', ls=':', lw=1)
ax2.set_xlabel('redshift z')
ax2.set_ylabel('D_TIFA / D_LCDM')
ax2.set_title('Growth suppression relative to ΛCDM')
ax2.grid(True)
ax2.set_xlim(0, 3)

plt.tight_layout()
plt.savefig('growth_factor.png', dpi=150)
plt.show()

In [ ]:

import numpy as np
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

# ─────────────────────────────────────────────
# 1. Build background ONCE (coarse but enough)
# ─────────────────────────────────────────────

f_E = 1.0
phi_ini = 0.3908 * np.pi * f_E
L4, sol = find_Lambda4_thawing(phi_ini)
Lambda4_global = L4

# Use moderate resolution (500 points enough)
N_arr = np.linspace(sol.t[0], 0.0, 500)
a_arr = np.exp(N_arr)

H_arr = []
Omega_m0 = 0.315

for N in N_arr:
    ph, dph = sol.sol(N)
    H2 = H_sq(np.exp(N), ph, dph)
    H_arr.append(np.sqrt(abs(H2)))

H_arr = np.array(H_arr)

# Interpolators (linear is enough)
H_interp = interp1d(a_arr, H_arr, kind='linear',
                    fill_value='extrapolate')

def Omega_m(a):
    return Omega_m0 / (a**3 * H_interp(a)**2)

# ─────────────────────────────────────────────
# 2. Growth ODE in x = ln a  (FAST + STABLE)
# ─────────────────────────────────────────────

# y = [D, dD/dx]
# d^2D/dx^2 + [2 + dlnH/dx] dD/dx - 3/2 Ωm(a) D = 0

def growth_ode_ln_a(x, y):
    D, Dp = y
    a = np.exp(x)

    H = H_interp(a)

    # analytic derivative via small step in x (cheap)
    dx = 1e-4
    a_p = np.exp(x + dx)
    a_m = np.exp(x - dx)
    H_p = H_interp(a_p)
    H_m = H_interp(a_m)

    dlnH_dx = (np.log(H_p) - np.log(H_m)) / (2 * dx)

    Om = Omega_m(a)

    Dpp = -(2 + dlnH_dx) * Dp + 1.5 * Om * D
    return [Dp, Dpp]

# Initial conditions deep in matter era
x_ini = np.log(1/51)
x_end = 0.0

# In matter era: D ∝ a → D = e^x, D' = D
D_ini = np.exp(x_ini)
Dp_ini = D_ini

x_eval = np.linspace(x_ini, x_end, 600)

sol_growth = solve_ivp(
    growth_ode_ln_a,
    [x_ini, x_end],
    [D_ini, Dp_ini],
    t_eval=x_eval,
    rtol=1e-6,
    atol=1e-8
)

D_today = sol_growth.y[0, -1]

# ─────────────────────────────────────────────
# 3. Fast LCDM analytic background
# ─────────────────────────────────────────────

Omega_L = 1.0 - Omega_m0

def H_LCDM(a):
    return np.sqrt(Omega_m0/a**3 + Omega_L)

def Omega_m_LCDM(a):
    return Omega_m0/(a**3 * H_LCDM(a)**2)

def growth_LCDM(x, y):
    D, Dp = y
    a = np.exp(x)

    H = H_LCDM(a)

    dx = 1e-4
    H_p = H_LCDM(np.exp(x+dx))
    H_m = H_LCDM(np.exp(x-dx))
    dlnH_dx = (np.log(H_p)-np.log(H_m))/(2*dx)

    Om = Omega_m_LCDM(a)

    Dpp = -(2 + dlnH_dx) * Dp + 1.5 * Om * D
    return [Dp, Dpp]

sol_LCDM = solve_ivp(
    growth_LCDM,
    [x_ini, 0.0],
    [D_ini, Dp_ini],
    t_eval=x_eval,
    rtol=1e-6,
    atol=1e-8
)

D_LCDM_today = sol_LCDM.y[0, -1]

# ─────────────────────────────────────────────
# 4. Final observables
# ─────────────────────────────────────────────

ratio = D_today / D_LCDM_today

sigma8_LCDM = 0.811
sigma8_TIFA = sigma8_LCDM * ratio

S8_LCDM = sigma8_LCDM * np.sqrt(Omega_m0/0.3)
S8_TIFA = sigma8_TIFA  * np.sqrt(Omega_m0/0.3)

print("\n── FAST Growth Results ──")
print(f"D ratio            : {ratio:.5f}")
print(f"sigma8 TIFA        : {sigma8_TIFA:.4f}")
print(f"S8 TIFA            : {S8_TIFA:.4f}")
print(f"ΔS8                : {S8_TIFA - S8_LCDM:.4f}")

NameError: name 'find_Lambda4_thawing' is not defined